<a href="https://colab.research.google.com/github/almagomen/v2v/blob/main/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Ejecuta las celdas en orden.
3. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `LMNT_API_KEY`

## 1. Clonar repo e instalar dependencias

In [ ]:
# @title ⚠️ Un solo paso: clonar monorepo + construir OpenReel + levantar API (con túnel)

import os, time, threading, sys, warnings
from pathlib import Path
from google.colab import userdata, drive
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- CONFIGURACIÓN ---
REPO_PATH = Path("/content/v2v")
BRANCH_NAME = "monorepo-subtree"  # rama que contiene OpenReel como subtree
BACKEND_PORT = 8000

GHP_TOKEN = userdata.get("GHP_GITHUB_TOKEN")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git" if GHP_TOKEN else "https://github.com/almagomen/v2v.git"

# --- Drive (opcional) ---
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# --- Clonar/actualizar repo ---
if not REPO_PATH.exists():
    print("📥 Clonando repositorio...")
    !git clone -q -b {BRANCH_NAME} {REPO_URL} {REPO_PATH}
else:
    print("🔄 Actualizando repositorio...")
    !git -C {REPO_PATH} fetch origin {BRANCH_NAME} -q
    !git -C {REPO_PATH} checkout {BRANCH_NAME} -q
    !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# --- System deps ---
print("🧰 Instalando dependencias del sistema (ffmpeg)...")
!apt-get update -qq
!apt-get install -y ffmpeg -qq

# --- Python deps ---
print("🐍 Instalando dependencias Python...")
!pip install -r {REPO_PATH/'requirements.txt'} -q

# --- Node/pnpm deps ---
print("🟦 Instalando Node + pnpm...")
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs -qq
!npm install -g pnpm -q

# --- Build OpenReel (subtree ya está en el repo) ---
OPENREEL_PATH = REPO_PATH / "openreel"
print("🏗️  Construyendo OpenReel (WASM + web)...")
!cd {OPENREEL_PATH} && pnpm install -q
!cd {OPENREEL_PATH/'packages'/'core'} && npm install -q
!cd {OPENREEL_PATH} && pnpm build

# --- Levantar backend FastAPI (sirve / y /api) ---
import uvicorn

class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self):
        return


def run_backend():
    from api.main import app
    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_level="info")
    server = ControlledServer(config=config)
    server.run()

print(f"🚀 Iniciando backend en 0.0.0.0:{BACKEND_PORT}...")
thread = threading.Thread(target=run_backend, daemon=True)
thread.start()

time.sleep(4)

# --- Túnel público (ngrok) ---
print("🌐 Creando túnel público...")
!pip install pyngrok -q
from pyngrok import ngrok

public = ngrok.connect(BACKEND_PORT)
print("\n✅ Abre OpenReel aquí:")
print(public.public_url)
print("\n- Docs API:")
print(public.public_url + "/docs")
